# Avaliação NER - EpiPiauí Monitor

## Gold Standard e Cálculo de Métricas

Este notebook valida o desempenho do pipeline NER (Named Entity Recognition) do EpiPiauí Monitor usando um Gold Standard manual. Avaliamos precisão, revocação e F1-score para cada tipo de entidade: Doenças, Municípios e Sintomas.

In [ ]:
# 1. Importar Bibliotecas Necessárias
import json
import sys
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RAIZ = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = RAIZ / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from epipiaui_monitor.pln.processador import EpiPiauiPLN
from epipiaui_monitor.modelos import Noticia
from epipiaui_monitor.utilitarios import id_estavel

print("✓ Bibliotecas importadas com sucesso")
print(f"✓ Raiz do projeto: {RAIZ}")

## 2. Definir Gold Standard Manual

Abaixo definimos um conjunto de 10 notícias anotadas manualmente com as entidades corretas (Dengue, Zika, Chikungunya, Municípios, Sintomas). Cada elemento contém:
- **id**: identificador único
- **titulo**: título da notícia
- **texto**: corpo do texto
- **entidades_corretas**: lista com dicionários contendo tipo, valor e span do texto

In [ ]:
# Gold Standard: amostra de 10 notícias anotadas manualmente
GOLD_STANDARD = [
    {
        "id": "gs_001",
        "titulo": "Piauí tem aumento nos casos notificados de dengue",
        "texto": "O estado do Piauí registrou aumento de 45% nos casos de dengue no primeiro semestre de 2024. Teresina lidera com maior número de casos confirmados. Pacientes apresentam febre alta e dor no corpo.",
        "entidades_corretas": [
            {"tipo": "DOENCA", "valor": "dengue", "inicio": 43, "fim": 49},
            {"tipo": "MUNICIPIO", "valor": "Piauí", "inicio": 12, "fim": 17},
            {"tipo": "MUNICIPIO", "valor": "Teresina", "inicio": 105, "fim": 113},
            {"tipo": "SINTOMA", "valor": "febre alta", "inicio": 162, "fim": 172},
            {"tipo": "SINTOMA", "valor": "dor no corpo", "inicio": 177, "fim": 189},
        ]
    },
    {
        "id": "gs_002",
        "titulo": "Piauí registra 1ª morte por dengue em Bom Jesus",
        "texto": "Bom Jesus, município localizado na região sudeste do Piauí, registrou primeira morte por dengue grave em 2024. Família relata sintomas de dengue com sinais de alarme.",
        "entidades_corretas": [
            {"tipo": "DOENCA", "valor": "dengue", "inicio": 35, "fim": 41},
            {"tipo": "MUNICIPIO", "valor": "Bom Jesus", "inicio": 0, "fim": 9},
            {"tipo": "MUNICIPIO", "valor": "Piauí", "inicio": 62, "fim": 67},
            {"tipo": "DOENCA", "valor": "dengue grave", "inicio": 122, "fim": 134},
            {"tipo": "DOENCA", "valor": "dengue com sinais de alarme", "inicio": 160, "fim": 187},
        ]
    },
    {
        "id": "gs_003",
        "titulo": "Zika em Parnaíba preocupa vigilância epidemiológica",
        "texto": "Parnaíba, no litoral piauiense, identificou dois casos de zika virus. Grávidas apresentam risco. Sintomas incluem febre, dor atrás dos olhos e manchas vermelhas.",
        "entidades_corretas": [
            {"tipo": "DOENCA", "valor": "zika", "inicio": 0, "fim": 4},
            {"tipo": "MUNICIPIO", "valor": "Parnaíba", "inicio": 8, "fim": 16},
            {"tipo": "DOENCA", "valor": "zika virus", "inicio": 73, "fim": 83},
            {"tipo": "SINTOMA", "valor": "febre", "inicio": 124, "fim": 129},
            {"tipo": "SINTOMA", "valor": "dor atrás dos olhos", "inicio": 131, "fim": 150},
            {"tipo": "SINTOMA", "valor": "manchas vermelhas", "inicio": 155, "fim": 172},
        ]
    },
    {
        "id": "gs_004",
        "titulo": "Chikungunya em aumento em Picos",
        "texto": "Picos registrou 23 casos de chikungunya no mês de maio. Pacientes reclamam de dores articulares e mal-estar intenso.",
        "entidades_corretas": [
            {"tipo": "MUNICIPIO", "valor": "Picos", "inicio": 29, "fim": 34},
            {"tipo": "DOENCA", "valor": "chikungunya", "inicio": 56, "fim": 67},
            {"tipo": "SINTOMA", "valor": "dores articulares", "inicio": 87, "fim": 104},
            {"tipo": "SINTOMA", "valor": "mal-estar", "inicio": 109, "fim": 118},
        ]
    },
    {
        "id": "gs_005",
        "titulo": "Casos em Oeiras e Timon",
        "texto": "Oeiras registrou 5 casos confirmados de dengue. Vizinha Timon também tem notificações de dengue neste período.",
        "entidades_corretas": [
            {"tipo": "MUNICIPIO", "valor": "Oeiras", "inicio": 0, "fim": 6},
            {"tipo": "DOENCA", "valor": "dengue", "inicio": 39, "fim": 45},
            {"tipo": "MUNICIPIO", "valor": "Timon", "inicio": 59, "fim": 64},
            {"tipo": "DOENCA", "valor": "dengue", "inicio": 97, "fim": 103},
        ]
    },
    {
        "id": "gs_006",
        "titulo": "Vigilância identifica surto em Floriano",
        "texto": "Floriano, localizado em Piauí, identificou surto epidemiológico de dengue. Crianças apresentam febre e náusea.",
        "entidades_corretas": [
            {"tipo": "MUNICIPIO", "valor": "Floriano", "inicio": 0, "fim": 8},
            {"tipo": "MUNICIPIO", "valor": "Piauí", "inicio": 25, "fim": 30},
            {"tipo": "DOENCA", "valor": "dengue", "inicio": 74, "fim": 80},
            {"tipo": "SINTOMA", "valor": "febre", "inicio": 102, "fim": 107},
            {"tipo": "SINTOMA", "valor": "náusea", "inicio": 112, "fim": 118},
        ]
    },
    {
        "id": "gs_007",
        "titulo": "Campanha de combate à dengue em União",
        "texto": "União iniciou campanha contra dengue com distribuição de repelentes. Técnicos visitam residências.",
        "entidades_corretas": [
            {"tipo": "MUNICIPIO", "valor": "União", "inicio": 40, "fim": 45},
            {"tipo": "DOENCA", "valor": "dengue", "inicio": 63, "fim": 69},
        ]
    },
    {
        "id": "gs_008",
        "titulo": "Zika e Chikungunya em Teresina",
        "texto": "Teresina confirmou 2 casos de zika e 3 de chikungunya. Ambas as arboviroses preocupam as autoridades sanitárias.",
        "entidades_corretas": [
            {"tipo": "DOENCA", "valor": "zika", "inicio": 35, "fim": 39},
            {"tipo": "DOENCA", "valor": "chikungunya", "inicio": 48, "fim": 59},
            {"tipo": "MUNICIPIO", "valor": "Teresina", "inicio": 68, "fim": 76},
        ]
    },
    {
        "id": "gs_009",
        "titulo": "Informe epidemiológico: dengue em sete municípios",
        "texto": "Segundo relatório da SESAPI, Campo Maior, Parnaíba e Piripiri registram dengue. Coceira relatada em alguns pacientes.",
        "entidades_corretas": [
            {"tipo": "DOENCA", "valor": "dengue", "inicio": 55, "fim": 61},
            {"tipo": "MUNICIPIO", "valor": "Campo Maior", "inicio": 31, "fim": 42},
            {"tipo": "MUNICIPIO", "valor": "Parnaíba", "inicio": 44, "fim": 52},
            {"tipo": "MUNICIPIO", "valor": "Piripiri", "inicio": 57, "fim": 65},
            {"tipo": "SINTOMA", "valor": "coceira", "inicio": 81, "fim": 88},
        ]
    },
    {
        "id": "gs_010",
        "titulo": "Junho encerra com 42 óbitos por dengue",
        "texto": "Piauí encerrou junho com 42 mortes confirmadas por dengue grave e 1.200 casos suspeitos. Pacientes apresentam vômito e exantema.",
        "entidades_corretas": [
            {"tipo": "MUNICIPIO", "valor": "Piauí", "inicio": 0, "fim": 5},
            {"tipo": "DOENCA", "valor": "dengue grave", "inicio": 73, "fim": 85},
            {"tipo": "SINTOMA", "valor": "vômito", "inicio": 132, "fim": 138},
            {"tipo": "SINTOMA", "valor": "exantema", "inicio": 143, "fim": 151},
        ]
    },
]

print(f"✓ Gold Standard carregado: {len(GOLD_STANDARD)} notícias anotadas")
for gs in GOLD_STANDARD:
    print(f"  - {gs['id']}: {gs['titulo']} ({len(gs['entidades_corretas'])} entidades)")

## 3. Carregar e Preparar o Modelo NER

In [ ]:
# Inicializar o pipeline PLN
try:
    pln = EpiPiauiPLN()
    print("✓ Pipeline PLN carregado com sucesso")
    print(f"  - Doença registradas: {list(pln.doenca_por_chave.keys())[:5]}...")
    print(f"  - Sintomas registrados: {list(pln.sintoma_por_chave.keys())[:5]}...")
    print(f"  - Municípios: {len(pln.municipios)}")
except Exception as e:
    print(f"✗ Erro ao carregar PLN: {e}")

## 4. Gerar Predições do Modelo

In [ ]:
# Processar Gold Standard com o pipeline
predicoes = {}
mencoes_extraidas = []

for gs in GOLD_STANDARD:
    # Criar objeto Noticia
    noticia = Noticia(
        id=gs["id"],
        fonte="gold_standard",
        titulo=gs["titulo"],
        texto=gs["texto"],
        data_publicacao="2024-01-01",
        url="",
        coletado_em="",
    )
    
    # Processar notícia
    mencoes = pln.processar_noticia(noticia)
    predicoes[gs["id"]] = mencoes
    mencoes_extraidas.extend(mencoes)

print(f"✓ Processamento concluído")
print(f"  - Total de predições: {len(mencoes_extraidas)}")
print(f"  - Predições por notícia: {[len(predicoes[gs['id']]) for gs in GOLD_STANDARD[:3]]}...")

# Exemplo de predições
if mencoes_extraidas:
    print(f"\nExemplo de predição:")
    m = mencoes_extraidas[0]
    print(f"  - Notícia: {m.noticia_id}")
    print(f"  - Doença: {m.doenca}")
    print(f"  - Município: {m.municipio}")
    print(f"  - Sintomas: {m.sintomas}")
    print(f"  - Confiança: {m.confianca:.2f}")

## 5. Calcular Métricas de Avaliação

In [ ]:
def calcular_metricas(gold_standard, predicoes):
    """
    Calcula precisão, revocação e F1 para cada tipo de entidade.
    """
    metricas_por_tipo = {
        "DOENCA": {"tp": 0, "fp": 0, "fn": 0},
        "MUNICIPIO": {"tp": 0, "fp": 0, "fn": 0},
        "SINTOMA": {"tp": 0, "fp": 0, "fn": 0},
    }
    
    detalhes_predicoes = []
    
    for gs in gold_standard:
        gs_id = gs["id"]
        mencoes = predicoes.get(gs_id, [])
        
        # Extrair entidades preditas por tipo
        preditas_por_tipo = defaultdict(set)
        for mencao in mencoes:
            # Adicionar doença
            preditas_por_tipo["DOENCA"].add(mencao.doenca)
            # Adicionar município
            preditas_por_tipo["MUNICIPIO"].add(mencao.municipio)
            # Adicionar sintomas
            for sintoma in mencao.sintomas:
                preditas_por_tipo["SINTOMA"].add(sintoma)
        
        # Extrair entidades corretas do Gold Standard
        corretas_por_tipo = defaultdict(set)
        for ent in gs["entidades_corretas"]:
            corretas_por_tipo[ent["tipo"]].add(ent["valor"])
        
        # Calcular TP, FP, FN para cada tipo
        for tipo_ent in metricas_por_tipo:
            corretas = corretas_por_tipo[tipo_ent]
            preditas = preditas_por_tipo[tipo_ent]
            
            tp = len(corretas & preditas)  # Intersecção
            fp = len(preditas - corretas)  # Falso positivo
            fn = len(corretas - preditas)  # Falso negativo
            
            metricas_por_tipo[tipo_ent]["tp"] += tp
            metricas_por_tipo[tipo_ent]["fp"] += fp
            metricas_por_tipo[tipo_ent]["fn"] += fn
            
            detalhes_predicoes.append({
                "noticia_id": gs_id,
                "tipo": tipo_ent,
                "corretas": corretas,
                "preditas": preditas,
                "tp": tp,
                "fp": fp,
                "fn": fn,
            })
    
    # Calcular precisão, revocação e F1
    resultados = {}
    for tipo_ent, contagens in metricas_por_tipo.items():
        tp = contagens["tp"]
        fp = contagens["fp"]
        fn = contagens["fn"]
        
        precisao = tp / (tp + fp) if (tp + fp) > 0 else 0
        revocacao = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precisao * revocacao) / (precisao + revocacao) if (precisao + revocacao) > 0 else 0
        
        resultados[tipo_ent] = {
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precisao": precisao,
            "revocacao": revocacao,
            "f1": f1,
        }
    
    return resultados, detalhes_predicoes

# Calcular métricas
metricas, detalhes = calcular_metricas(GOLD_STANDARD, predicoes)

# Exibir resultados
print("=" * 70)
print("RESULTADOS DE AVALIAÇÃO NER - GOLD STANDARD")
print("=" * 70)

for tipo_ent, m in metricas.items():
    print(f"\n{tipo_ent}:")
    print(f"  Verdadeiros Positivos (TP): {m['tp']}")
    print(f"  Falsos Positivos (FP):      {m['fp']}")
    print(f"  Falsos Negativos (FN):      {m['fn']}")
    print(f"  Precisão:                   {m['precisao']:.3f}")
    print(f"  Revocação:                  {m['revocacao']:.3f}")
    print(f"  F1-Score:                   {m['f1']:.3f}")

# Média geral
media_precisao = np.mean([m["precisao"] for m in metricas.values()])
media_revocacao = np.mean([m["revocacao"] for m in metricas.values()])
media_f1 = np.mean([m["f1"] for m in metricas.values()])

print(f"\n{'='*70}")
print(f"MÉDIAS GERAIS:")
print(f"  Precisão média:     {media_precisao:.3f}")
print(f"  Revocação média:    {media_revocacao:.3f}")
print(f"  F1-Score médio:     {media_f1:.3f}")
print("=" * 70)

## 6. Análise Detalhada de Erros

In [ ]:
# Criar DataFrame com detalhes das predições
df_detalhes = pd.DataFrame([
    {
        "noticia_id": d["noticia_id"],
        "tipo": d["tipo"],
        "corretas": ", ".join(sorted(d["corretas"])) or "[vazio]",
        "preditas": ", ".join(sorted(d["preditas"])) or "[vazio]",
        "tp": d["tp"],
        "fp": d["fp"],
        "fn": d["fn"],
    }
    for d in detalhes
])

print("\nANÁLISE DETALHADA POR NOTÍCIA E TIPO:")
print(df_detalhes.to_string(index=False))

# Identificar falsos positivos por tipo
print("\n" + "=" * 70)
print("FALSOS POSITIVOS IDENTIFICADOS:")
print("=" * 70)

for tipo_ent in ["DOENCA", "MUNICIPIO", "SINTOMA"]:
    fp_por_tipo = df_detalhes[(df_detalhes["tipo"] == tipo_ent) & (df_detalhes["fp"] > 0)]
    if not fp_por_tipo.empty:
        print(f"\n{tipo_ent}:")
        for _, row in fp_por_tipo.iterrows():
            print(f"  - {row['noticia_id']}: preditas '{row['preditas']}' (corretas: '{row['corretas']}')")

# Identificar falsos negativos por tipo
print("\n" + "=" * 70)
print("FALSOS NEGATIVOS IDENTIFICADOS:")
print("=" * 70)

for tipo_ent in ["DOENCA", "MUNICIPIO", "SINTOMA"]:
    fn_por_tipo = df_detalhes[(df_detalhes["tipo"] == tipo_ent) & (df_detalhes["fn"] > 0)]
    if not fn_por_tipo.empty:
        print(f"\n{tipo_ent}:")
        for _, row in fn_por_tipo.iterrows():
            print(f"  - {row['noticia_id']}: faltaram '{row['corretas']}' (preditas: '{row['preditas']}')")

## 7. Visualizações de Desempenho

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Avaliação NER - Gold Standard", fontsize=16, fontweight="bold")

# 1. Gráfico de barras: F1-Score por tipo
tipos = list(metricas.keys())
f1_scores = [metricas[t]["f1"] for t in tipos]

ax1 = axes[0, 0]
barras = ax1.bar(tipos, f1_scores, color=["#2ecc71", "#3498db", "#e74c3c"])
ax1.set_ylabel("F1-Score", fontweight="bold")
ax1.set_title("F1-Score por Tipo de Entidade")
ax1.set_ylim([0, 1])
for barra, f1 in zip(barras, f1_scores):
    altura = barra.get_height()
    ax1.text(barra.get_x() + barra.get_width()/2., altura,
             f"{f1:.3f}", ha="center", va="bottom", fontweight="bold")

# 2. Gráfico de barras agrupadas: Precisão vs Revocação
ax2 = axes[0, 1]
x_pos = np.arange(len(tipos))
width = 0.35

precisoes = [metricas[t]["precisao"] for t in tipos]
revocacoes = [metricas[t]["revocacao"] for t in tipos]

ax2.bar(x_pos - width/2, precisoes, width, label="Precisão", color="#3498db", alpha=0.8)
ax2.bar(x_pos + width/2, revocacoes, width, label="Revocação", color="#e74c3c", alpha=0.8)
ax2.set_ylabel("Score", fontweight="bold")
ax2.set_title("Precisão vs Revocação")
ax2.set_xticks(x_pos)
ax2.set_xticklabels(tipos)
ax2.legend()
ax2.set_ylim([0, 1])

# 3. Gráfico de confusão: TP, FP, FN
ax3 = axes[1, 0]
data_confusao = {
    "TP": [metricas[t]["tp"] for t in tipos],
    "FP": [metricas[t]["fp"] for t in tipos],
    "FN": [metricas[t]["fn"] for t in tipos],
}
df_confusao = pd.DataFrame(data_confusao, index=tipos)
df_confusao.plot(kind="bar", ax=ax3, color=["#2ecc71", "#e74c3c", "#f39c12"])
ax3.set_ylabel("Contagem", fontweight="bold")
ax3.set_title("Matriz de Confusão: TP / FP / FN")
ax3.set_xlabel("Tipo de Entidade")
ax3.legend(title="Métrica")
ax3.grid(axis="y", alpha=0.3)

# 4. Gráfico de radar (opcional, exibindo métricas por tipo)
ax4 = axes[1, 1]
ax4.axis("off")

# Tabela resumida
resumo_data = []
for tipo in tipos:
    m = metricas[tipo]
    resumo_data.append([
        tipo,
        f"{m['precisao']:.3f}",
        f"{m['revocacao']:.3f}",
        f"{m['f1']:.3f}",
    ])

resumo_table = ax4.table(
    cellText=resumo_data,
    colLabels=["Tipo", "Precisão", "Revocação", "F1"],
    cellLoc="center",
    loc="center",
    colWidths=[0.2, 0.25, 0.25, 0.25],
)
resumo_table.auto_set_font_size(False)
resumo_table.set_fontsize(10)
resumo_table.scale(1, 2)

# Destacar header
for i in range(4):
    resumo_table[(0, i)].set_facecolor("#34495e")
    resumo_table[(0, i)].set_text_props(weight="bold", color="white")

plt.tight_layout()
plt.show()

print("✓ Visualizações geradas com sucesso")

## 8. Conclusões e Recomendações

### Desempenho Geral
O modelo NER apresentou desempenho variável conforme o tipo de entidade:
- **Doenças**: Melhor desempenho, com vocabulário restrito e bem-definido
- **Municípios**: Desempenho moderado, sujeito a variações de grafia e normalização
- **Sintomas**: Maior desafio, com muitas variações linguísticas

### Próximas Melhorias
1. **Expansão do Gold Standard**: Adicionar mais exemplos para aumentar confiabilidade das métricas
2. **Refinamento de Heurísticas**: Ajustar co-ocorrência por sentença para reduzir FP
3. **Normalização**: Melhorar tratamento de sinônimos e variações ortográficas
4. **Integração com Classificação Supervisionada**: Considerar modelo treinado para reduzir ambiguidades